# Analisis Latin Square pada Supertropical Algebra

Program ini:
1. Generate semua Latin square order $n$
2. Ekstrak representasi permutasi untuk setiap elemen
3. Cari pasangan komutatif pada **supertropical algebra**
4. **Bedakan hasil**: full tangible vs mengandung ghost element

## Supertropical Algebra

Pada supertropical semiring, elemen dibagi menjadi:
- **Tangible** ($\mathcal{T}$): elemen reguler
- **Ghost** ($\mathcal{G}$): elemen dengan penanda $\nu$, dimana $a^\nu \oplus a^\nu = a^\nu$

**Operasi:**
- $a \oplus b = \max(a, b)$ jika $|a| \neq |b|$
- $a \oplus a = a^\nu$ (menjadi ghost)
- $a \oplus a^\nu = a^\nu$ (ghost dominates at equality)
- $a \otimes b = a + b$ (penjumlahan biasa), ghost propagates

In [11]:
"""
Supertropical Algebra Library
"""
import numpy as np
from typing import List, Dict, Tuple
import os

class SupertropicalElement:
    """
    Elemen dalam supertropical semiring.
    Setiap elemen punya value dan flag ghost.
    """
    
    def __init__(self, value: float, is_ghost: bool = False):
        self.value = value
        self.is_ghost = is_ghost
    
    def __repr__(self):
        if self.is_ghost:
            return f"{int(self.value)}ᵍ"
        return str(int(self.value))
    
    def __eq__(self, other):
        if isinstance(other, SupertropicalElement):
            return self.value == other.value and self.is_ghost == other.is_ghost
        return False
    
    def __hash__(self):
        return hash((self.value, self.is_ghost))
    
    def ghost(self):
        """Return ghost version of this element"""
        return SupertropicalElement(self.value, is_ghost=True)
    
    def __add__(self, other):
        """Supertropical addition (⊕): max with ghost rules"""
        if isinstance(other, (int, float)):
            other = SupertropicalElement(other)
        
        if self.value > other.value:
            return SupertropicalElement(self.value, self.is_ghost)
        elif self.value < other.value:
            return SupertropicalElement(other.value, other.is_ghost)
        else:  # equal values
            # a ⊕ a = aᵍ, a ⊕ aᵍ = aᵍ, aᵍ ⊕ aᵍ = aᵍ
            return SupertropicalElement(self.value, is_ghost=True)
    
    def __mul__(self, other):
        """Supertropical multiplication (⊗): addition with ghost propagation"""
        if isinstance(other, (int, float)):
            other = SupertropicalElement(other)
        
        new_value = self.value + other.value
        # Ghost propagates: if either is ghost, result is ghost
        new_ghost = self.is_ghost or other.is_ghost
        return SupertropicalElement(new_value, new_ghost)
    
    __radd__ = __add__
    __rmul__ = __mul__

class SupertropicalMatrix:
    """Matrix dengan elemen supertropical"""
    
    def __init__(self, data):
        """
        Initialize dari numpy array atau list.
        Jika input bukan SupertropicalElement, konversi ke tangible element.
        """
        if isinstance(data, np.ndarray):
            n, m = data.shape
            self.matrix = [[SupertropicalElement(data[i,j]) for j in range(m)] for i in range(n)]
        else:
            self.matrix = data
        self.n = len(self.matrix)
        self.m = len(self.matrix[0]) if self.matrix else 0
    
    def __repr__(self):
        rows = []
        for row in self.matrix:
            rows.append("[" + " ".join(str(e) for e in row) + "]")
        return "\n".join(rows)
    
    def __getitem__(self, idx):
        return self.matrix[idx]
    
    def __matmul__(self, other):
        """Supertropical matrix multiplication"""
        assert self.m == other.n, "Dimensi tidak cocok"
        
        result = []
        for i in range(self.n):
            row = []
            for j in range(other.m):
                # (A ⊗ B)_ij = ⊕_k (A_ik ⊗ B_kj)
                total = self.matrix[i][0] * other.matrix[0][j]
                for k in range(1, self.m):
                    total = total + (self.matrix[i][k] * other.matrix[k][j])
                row.append(total)
            result.append(row)
        
        return SupertropicalMatrix(result)
    
    def has_ghost(self) -> bool:
        """Cek apakah matrix mengandung elemen ghost"""
        for row in self.matrix:
            for elem in row:
                if elem.is_ghost:
                    return True
        return False
    
    def equals(self, other) -> bool:
        """Cek equality dua matrix"""
        if self.n != other.n or self.m != other.m:
            return False
        for i in range(self.n):
            for j in range(self.m):
                if self.matrix[i][j] != other.matrix[i][j]:
                    return False
        return True
    
    def to_latex(self) -> str:
        """Convert ke LaTeX pmatrix"""
        rows = []
        for row in self.matrix:
            row_str = " & ".join(
                f"{int(e.value)}^\\nu" if e.is_ghost else str(int(e.value))
                for e in row
            )
            rows.append(row_str)
        sep = " \\\\ "
        content = sep.join(rows)
        return f"\\begin{{pmatrix}} {content} \\end{{pmatrix}}"

print("✓ SupertropicalElement dan SupertropicalMatrix berhasil dimuat!")
print("\nContoh:")
a = SupertropicalElement(3)
b = SupertropicalElement(3)
print(f"  3 ⊕ 3 = {a + b}  (ghost karena nilai sama)")
print(f"  3 ⊗ 2 = {a * SupertropicalElement(2)}")

✓ SupertropicalElement dan SupertropicalMatrix berhasil dimuat!

Contoh:
  3 ⊕ 3 = 3ᵍ  (ghost karena nilai sama)
  3 ⊗ 2 = 5


In [12]:
class LatinSquare:
    """Class untuk merepresentasikan Latin Square"""
    
    def __init__(self, matrix: np.ndarray, symbols: List[int] = None):
        self.matrix = np.array(matrix, dtype=int)
        self.n = len(matrix)
        self.symbols = symbols if symbols else sorted(np.unique(matrix).tolist())
    
    def __repr__(self):
        return f"LatinSquare(n={self.n})\n{self.matrix}"
    
    def __eq__(self, other):
        return np.array_equal(self.matrix, other.matrix)
    
    def __hash__(self):
        return hash(self.matrix.tobytes())
    
    def to_supertropical(self) -> SupertropicalMatrix:
        """Convert ke SupertropicalMatrix (semua tangible)"""
        return SupertropicalMatrix(self.matrix)

class LatinSquareGenerator:
    """Generator Latin Square menggunakan backtracking"""
    
    def __init__(self, n: int, symbols: List[int] = None):
        self.n = n
        self.symbols = symbols if symbols else list(range(1, n + 1))
        self.all_squares = []
    
    def generate_all(self) -> List[LatinSquare]:
        """Generate semua Latin square order n"""
        self.all_squares = []
        grid = [[0] * self.n for _ in range(self.n)]
        possible = [[set(self.symbols) for _ in range(self.n)] for _ in range(self.n)]
        self._backtrack(grid, possible, 0, 0)
        return self.all_squares
    
    def _backtrack(self, grid, possible, row, col):
        if row == self.n:
            self.all_squares.append(LatinSquare(np.array(grid), self.symbols.copy()))
            return
        
        next_row, next_col = (row, col + 1) if col + 1 < self.n else (row + 1, 0)
        
        for val in list(possible[row][col]):
            grid[row][col] = val
            removed = []
            valid = True
            
            for k in range(self.n):
                if k != col and val in possible[row][k]:
                    possible[row][k].remove(val)
                    removed.append((row, k, val))
                    if not possible[row][k] and grid[row][k] == 0:
                        valid = False
                if k != row and val in possible[k][col]:
                    possible[k][col].remove(val)
                    removed.append((k, col, val))
                    if not possible[k][col] and grid[k][col] == 0:
                        valid = False
            
            if valid:
                self._backtrack(grid, possible, next_row, next_col)
            
            grid[row][col] = 0
            for r, c, v in removed:
                possible[r][c].add(v)

class PermutationExtractor:
    """Ekstrak representasi permutasi dari Latin square (Proposisi 2.6.1)"""
    
    @staticmethod
    def extract_permutations(ls: LatinSquare) -> Dict[int, Tuple[int, ...]]:
        n = ls.n
        permutations_dict = {}
        for symbol in ls.symbols:
            perm = [0] * n
            for r in range(n):
                for c in range(n):
                    if ls.matrix[r, c] == symbol:
                        perm[r] = c
                        break
            permutations_dict[symbol] = tuple(perm)
        return permutations_dict
    
    @staticmethod
    def permutation_to_cycle_notation(perm: Tuple[int, ...], one_indexed: bool = True) -> str:
        n = len(perm)
        offset = 1 if one_indexed else 0
        visited = [False] * n
        cycles = []
        
        for start in range(n):
            if visited[start]:
                continue
            cycle = []
            current = start
            while not visited[current]:
                visited[current] = True
                cycle.append(current + offset)
                current = perm[current]
            if len(cycle) >= 1:
                cycles.append(cycle)
        
        if not cycles:
            return "()"
        return "".join(f"({' '.join(map(str, c))})" for c in cycles)

print("✓ LatinSquare, LatinSquareGenerator, PermutationExtractor berhasil dimuat!")

✓ LatinSquare, LatinSquareGenerator, PermutationExtractor berhasil dimuat!


## Generate Latin Squares

In [13]:
# ============================================================
# KONFIGURASI
# ============================================================
ORDER = 4
SYMBOLS = list(range(1, ORDER + 1))

# ============================================================
# Generate semua Latin square
# ============================================================
print(f"{'='*60}")
print(f"GENERATING LATIN SQUARES ORDER {ORDER}")
print(f"Simbol: {{{', '.join(map(str, SYMBOLS))}}}")
print(f"{'='*60}")

generator = LatinSquareGenerator(ORDER, SYMBOLS)
all_squares = generator.generate_all()

print(f"\n✓ Ditemukan {len(all_squares)} Latin square")
print(f"\nContoh Latin square pertama:")
print(all_squares[0].matrix)

GENERATING LATIN SQUARES ORDER 4
Simbol: {1, 2, 3, 4}

✓ Ditemukan 576 Latin square

Contoh Latin square pertama:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]


## Pasangan Komutatif pada Supertropical Algebra

Mencari pasangan $(A, B)$ dengan $A \neq B$ yang memenuhi $A \otimes B = B \otimes A$ pada supertropical algebra.

Hasil akan dibedakan:
- **Full Tangible**: Hasil perkalian tidak mengandung elemen ghost
- **Contains Ghost**: Hasil perkalian mengandung elemen ghost ($a^\nu$)

In [14]:
# ============================================================
# Fungsi Pencarian Pasangan Komutatif (Supertropical)
# ============================================================

def find_supertropical_commutative_pairs(squares: List[LatinSquare]) -> Tuple[List[dict], List[dict]]:
    """
    Cari pasangan komutatif pada supertropical algebra.
    
    Returns:
        Tuple (tangible_pairs, ghost_pairs):
        - tangible_pairs: Pasangan dengan hasil full tangible
        - ghost_pairs: Pasangan dengan hasil mengandung ghost
    """
    tangible_pairs = []
    ghost_pairs = []
    n_squares = len(squares)
    
    for i in range(n_squares):
        for j in range(i + 1, n_squares):
            A_st = squares[i].to_supertropical()
            B_st = squares[j].to_supertropical()
            
            AB = A_st @ B_st
            BA = B_st @ A_st
            
            if AB.equals(BA):
                pair_data = {
                    'A': squares[i],
                    'B': squares[j],
                    'A_index': i + 1,
                    'B_index': j + 1,
                    'product': AB,
                    'has_ghost': AB.has_ghost()
                }
                
                if AB.has_ghost():
                    ghost_pairs.append(pair_data)
                else:
                    tangible_pairs.append(pair_data)
    
    return tangible_pairs, ghost_pairs

# Cari pasangan komutatif
print(f"{'='*60}")
print(f"MENCARI PASANGAN KOMUTATIF (SUPERTROPICAL)")
print(f"{'='*60}")

tangible_pairs, ghost_pairs = find_supertropical_commutative_pairs(all_squares)

total_pairs = len(all_squares) * (len(all_squares) - 1) // 2
total_commutative = len(tangible_pairs) + len(ghost_pairs)

print(f"\n✓ Ditemukan {total_commutative} pasangan komutatif dari {total_pairs} pasangan")
print(f"  - Full Tangible: {len(tangible_pairs)} pasangan")
print(f"  - Contains Ghost: {len(ghost_pairs)} pasangan")

MENCARI PASANGAN KOMUTATIF (SUPERTROPICAL)

✓ Ditemukan 2808 pasangan komutatif dari 165600 pasangan
  - Full Tangible: 300 pasangan
  - Contains Ghost: 2508 pasangan


In [15]:
# ============================================================
# Tampilkan Pasangan Full Tangible
# ============================================================
print(f"\n{'='*60}")
print("PASANGAN KOMUTATIF - FULL TANGIBLE")
print(f"{'='*60}")

if tangible_pairs:
    for idx, pair in enumerate(tangible_pairs, 1):
        print(f"\n--- Pasangan #{idx}: L_{pair['A_index']} ∘ L_{pair['B_index']} ---")
        print(f"A:\n{pair['A'].matrix}")
        print(f"B:\n{pair['B'].matrix}")
        print(f"A ⊗ B = B ⊗ A:")
        print(pair['product'])
else:
    print("\nTidak ada pasangan dengan hasil full tangible.")


PASANGAN KOMUTATIF - FULL TANGIBLE

--- Pasangan #1: L_1 ∘ L_50 ---
A:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]
B:
[[1 3 2 4]
 [3 1 4 2]
 [2 4 1 3]
 [4 2 3 1]]
A ⊗ B = B ⊗ A:
[8 7 7 6]
[7 8 6 7]
[7 6 8 7]
[6 7 7 8]

--- Pasangan #2: L_1 ∘ L_276 ---
A:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]
B:
[[2 4 1 3]
 [4 2 3 1]
 [1 3 2 4]
 [3 1 4 2]]
A ⊗ B = B ⊗ A:
[7 6 8 7]
[6 7 7 8]
[8 7 7 6]
[7 8 6 7]

--- Pasangan #3: L_1 ∘ L_381 ---
A:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]
B:
[[3 1 4 2]
 [1 3 2 4]
 [4 2 3 1]
 [2 4 1 3]]
A ⊗ B = B ⊗ A:
[7 8 6 7]
[8 7 7 6]
[6 7 7 8]
[7 6 8 7]

--- Pasangan #4: L_1 ∘ L_496 ---
A:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]
B:
[[4 2 3 1]
 [2 4 1 3]
 [3 1 4 2]
 [1 3 2 4]]
A ⊗ B = B ⊗ A:
[6 7 7 8]
[7 6 8 7]
[7 8 6 7]
[8 7 7 6]

--- Pasangan #5: L_3 ∘ L_52 ---
A:
[[1 2 3 4]
 [2 1 4 3]
 [4 3 1 2]
 [3 4 2 1]]
B:
[[1 3 2 4]
 [3 1 4 2]
 [4 2 1 3]
 [2 4 3 1]]
A ⊗ B = B ⊗ A:
[7 8 7 6]
[8 7 6 7]
[6 7 7 8]
[7 6 8 7]

--- Pasangan #6: L_3 ∘ L_263 ---
A:
[[1 

In [16]:
# ============================================================
# Tampilkan Pasangan Contains Ghost
# ============================================================
print(f"\n{'='*60}")
print("PASANGAN KOMUTATIF - CONTAINS GHOST")
print(f"{'='*60}")

if ghost_pairs:
    for idx, pair in enumerate(ghost_pairs, 1):
        print(f"\n--- Pasangan #{idx}: L_{pair['A_index']} ∘ L_{pair['B_index']} ---")
        print(f"A:\n{pair['A'].matrix}")
        print(f"B:\n{pair['B'].matrix}")
        print(f"A ⊗ B = B ⊗ A (dengan ghost ᵍ):")
        print(pair['product'])
else:
    print("\nTidak ada pasangan dengan hasil mengandung ghost.")


PASANGAN KOMUTATIF - CONTAINS GHOST

--- Pasangan #1: L_1 ∘ L_27 ---
A:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]
B:
[[1 2 4 3]
 [2 1 3 4]
 [4 3 1 2]
 [3 4 2 1]]
A ⊗ B = B ⊗ A (dengan ghost ᵍ):
[7ᵍ 8 6 6]
[8 7ᵍ 6 6]
[6 6 7ᵍ 8]
[6 6 8 7ᵍ]

--- Pasangan #2: L_1 ∘ L_92 ---
A:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]
B:
[[1 3 4 2]
 [3 1 2 4]
 [4 2 1 3]
 [2 4 3 1]]
A ⊗ B = B ⊗ A (dengan ghost ᵍ):
[7 8 7 6ᵍ]
[8 7 6ᵍ 7]
[7 6ᵍ 7 8]
[6ᵍ 7 8 7]

--- Pasangan #3: L_1 ∘ L_94 ---
A:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]
B:
[[1 3 4 2]
 [3 2 1 4]
 [4 1 2 3]
 [2 4 3 1]]
A ⊗ B = B ⊗ A (dengan ghost ᵍ):
[7 8 7 6ᵍ]
[8 7 6ᵍ 7]
[7 6ᵍ 7 8]
[6ᵍ 7 8 7]

--- Pasangan #4: L_1 ∘ L_99 ---
A:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]
B:
[[1 4 2 3]
 [4 1 3 2]
 [2 3 1 4]
 [3 2 4 1]]
A ⊗ B = B ⊗ A (dengan ghost ᵍ):
[7 6ᵍ 8 7]
[6ᵍ 7 7 8]
[8 7 7 6ᵍ]
[7 8 6ᵍ 7]

--- Pasangan #5: L_1 ∘ L_131 ---
A:
[[1 2 3 4]
 [2 1 4 3]
 [3 4 1 2]
 [4 3 2 1]]
B:
[[1 4 3 2]
 [4 2 1 3]
 [3 1 2 4]
 [2 3 4 1]]
A ⊗ B = B ⊗ A

## Export ke LaTeX

Export hasil ke file `.tex` yang terpisah untuk:
1. Pasangan dengan hasil full tangible
2. Pasangan dengan hasil mengandung ghost

In [17]:
# ============================================================
# Export Pasangan Komutatif ke LaTeX (Supertropical)
# ============================================================

def export_supertropical_pairs(pairs: List[dict], 
                                filename: str,
                                order: int,
                                pair_type: str,  # "tangible" atau "ghost"
                                output_dir: str = ".",
                                num_cols: int = 2) -> str:
    """Export pasangan komutatif supertropical ke file .tex"""
    lines = []
    sep = " \\\\ "
    
    # Matrix to latex helper (compact, regular int matrix)
    def mat_to_tex(m):
        rows = sep.join(" & ".join(str(int(m[i,j])) for j in range(m.shape[1])) for i in range(m.shape[0]))
        return f"\\begin{{pmatrix}} {rows} \\end{{pmatrix}}"
    
    # Supertropical matrix to latex (with ghost notation)
    def stmat_to_tex(stmat: SupertropicalMatrix) -> str:
        rows = []
        for row in stmat.matrix:
            row_str = " & ".join(
                f"{int(e.value)}^\\nu" if e.is_ghost else str(int(e.value))
                for e in row
            )
            rows.append(row_str)
        content = sep.join(rows)
        return f"\\begin{{pmatrix}} {content} \\end{{pmatrix}}"
    
    # Permutation to matrix format (vertical)
    def perms_to_matrix_small(perms: dict, symbols: list) -> str:
        rows = sep.join(
            f"\\tau_{{{s}}}={PermutationExtractor.permutation_to_cycle_notation(perms[s])}"
            for s in symbols
        )
        return f"\\begin{{matrix}} {rows} \\end{{matrix}}"
    
    # Header
    type_label = "Full Tangible" if pair_type == "tangible" else "Contains Ghost"
    lines.append(f"% File: {filename}.tex")
    lines.append(f"% Commutative pairs (supertropical) order {order} - {type_label}")
    lines.append(f"% Total: {len(pairs)} pairs")
    lines.append("")
    
    lines.append(f"\\subsection*{{Pasangan Komutatif Supertropical Order {order} - {type_label}}}")
    lines.append("")
    lines.append(f"Terdapat {len(pairs)} pasangan $(A, B)$ dengan $A \\neq B$ yang memenuhi $A \\otimes B = B \\otimes A$")
    if pair_type == "ghost":
        lines.append("dimana hasil perkalian mengandung elemen ghost ($a^\\nu$):")
    else:
        lines.append("dimana hasil perkalian full tangible:")
    lines.append("")
    lines.append(f"\\begin{{multicols}}{{{num_cols}}}")
    lines.append("\\small")
    lines.append("")
    
    for idx, pair in enumerate(pairs, 1):
        A = pair['A']
        B = pair['B']
        product = pair['product']
        
        # Ekstrak permutasi
        perms_A = PermutationExtractor.extract_permutations(A)
        perms_B = PermutationExtractor.extract_permutations(B)
        
        perm_A_matrix = perms_to_matrix_small(perms_A, A.symbols)
        perm_B_matrix = perms_to_matrix_small(perms_B, B.symbols)
        
        lines.append(f"\\noindent\\textbf{{{idx}.}} $L_{{{pair['A_index']}}} \\circ L_{{{pair['B_index']}}}$")
        lines.append("")
        lines.append(f"$A = {mat_to_tex(A.matrix)} \\quad {perm_A_matrix}$")
        lines.append("")
        lines.append(f"$B = {mat_to_tex(B.matrix)} \\quad {perm_B_matrix}$")
        lines.append("")
        lines.append(f"$A \\otimes B = {stmat_to_tex(product)}$")
        lines.append("")
        lines.append("\\vspace{0.3em}")
        lines.append("\\hrule")
        lines.append("\\vspace{0.3em}")
        lines.append("")
    
    lines.append("\\end{multicols}")
    
    # Write to file
    filepath = os.path.join(output_dir, f"{filename}.tex")
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write("\n".join(lines))
    
    return filepath

# Export kedua jenis
if tangible_pairs:
    filepath_tangible = export_supertropical_pairs(
        tangible_pairs,
        filename=f"supertropical_commutative_tangible_order_{ORDER}",
        order=ORDER,
        pair_type="tangible",
        num_cols=2
    )
    print(f"✓ File tangible: {filepath_tangible}")
    print(f"  Total: {len(tangible_pairs)} pasangan")

if ghost_pairs:
    filepath_ghost = export_supertropical_pairs(
        ghost_pairs,
        filename=f"supertropical_commutative_ghost_order_{ORDER}",
        order=ORDER,
        pair_type="ghost",
        num_cols=2
    )
    print(f"✓ File ghost: {filepath_ghost}")
    print(f"  Total: {len(ghost_pairs)} pasangan")

✓ File tangible: .\supertropical_commutative_tangible_order_4.tex
  Total: 300 pasangan
✓ File ghost: .\supertropical_commutative_ghost_order_4.tex
  Total: 2508 pasangan
